In [ ]:
# @title Run Daily Options Screener (Earnings Filter + Correlation Removal)
# 1. INSTALL LIBRARIES
!pip install yfinance pandas_ta -q

import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from datetime import datetime, timedelta

# 2. DEFINE THE TICKER LIST
raw_tickers = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM
"""
ticker_list = [t.strip() for t in raw_tickers.split() if t.strip() and t.strip() != "XYZ"]

# 3. HELPER FUNCTIONS: EARNINGS & CORRELATION
def has_earnings_coming_up(ticker_symbol, days=15):
    try:
        t = yf.Ticker(ticker_symbol)
        cal = t.calendar
        earnings_dates = []
        
        if cal is None: return False
        elif isinstance(cal, dict):
            keys = cal.keys()
            for k in keys:
                if 'Earnings' in str(k):
                    earnings_dates = cal[k]
                    break
        
        if not earnings_dates: return False

        for date in earnings_dates:
             if hasattr(date, 'tzinfo') and date.tzinfo:
                 date = date.tz_localize(None)
             now = datetime.now()
             delta = date - now
             if 0 <= delta.days <= days:
                 return True
        return False
    except: return False

def filter_correlated_assets(candidate_list, price_data, max_items=10, threshold=0.85):
    """
    Selects top assets by volume, skipping any that are highly correlated (>0.85)
    to assets already selected.
    """
    if not candidate_list: return []
    
    # Sort candidates by Dollar Volume (Highest to Lowest)
    # We want to keep the most liquid one (e.g., SPY) and discard the copy (e.g., DIA)
    sorted_candidates = sorted(candidate_list, key=lambda x: x['DollarVolume'], reverse=True)
    
    selected = []
    
    for cand in sorted_candidates:
        if len(selected) >= max_items: break
        
        ticker = cand['Ticker']
        
        # If it's the first one, take it
        if not selected:
            selected.append(cand)
            continue
            
        # Check correlation against already selected assets
        is_duplicate = False
        cand_series = price_data[ticker]['Close'].pct_change().tail(60) # Use last 60 days returns
        
        for sel in selected:
            sel_ticker = sel['Ticker']
            sel_series = price_data[sel_ticker]['Close'].pct_change().tail(60)
            
            # Calculate correlation
            corr = cand_series.corr(sel_series)
            
            if corr > threshold:
                # Correlation too high
                is_duplicate = True
                break
        
        if not is_duplicate:
            selected.append(cand)
            
    return selected

# 4. DOWNLOAD DATA & FILTER LIQUIDITY
print(f"1/4: Downloading data for {len(ticker_list)} tickers...")
data = yf.download(ticker_list, period="6mo", group_by='ticker', auto_adjust=True, threads=True, progress=False)

print("2/4: Filtering Top 200 by Liquidity (Dollar Volume)...")
liquidity_stats = []
for ticker in ticker_list:
    try:
        if ticker not in data.columns.levels[0]: continue
        df = data[ticker].dropna()
        if df.empty: continue
        
        avg_price = df['Close'].tail(20).mean()
        avg_vol = df['Volume'].tail(20).mean()
        liquidity_stats.append({'Ticker': ticker, 'DollarVolume': avg_price * avg_vol})
    except: continue

liq_df = pd.DataFrame(liquidity_stats).sort_values(by='DollarVolume', ascending=False).head(200)
top_200_tickers = liq_df['Ticker'].tolist()

# 5. APPLY STRATEGY & EARNINGS FILTER
print("3/4: Screening Technicals & Checking Earnings...")
candidates = []

for ticker in top_200_tickers:
    try:
        df = data[ticker].dropna().copy()
        if len(df) < 50: continue

        # Indicators
        df['SMA_10'] = ta.sma(df['Close'], length=10)
        df['SMA_50'] = ta.sma(df['Close'], length=50)
        df['RSI'] = ta.rsi(df['Close'], length=14)
        current = df.iloc[-1]
        
        signal = None
        
        # Bullish Criteria
        if (current['Close'] > current['SMA_50']) and \
           (current['Close'] > current['SMA_10']) and \
           (40 < current['RSI'] < 65):
            signal = "BULLISH"

        # Bearish Criteria
        elif (current['Close'] < current['SMA_50']) and \
             (current['Close'] < current['SMA_10']) and \
             (35 < current['RSI'] < 60):
            signal = "BEARISH"
            
        if signal:
            if not has_earnings_coming_up(ticker, days=15):
                candidates.append({
                    'Ticker': ticker,
                    'Signal': signal,
                    'DollarVolume': liq_df[liq_df['Ticker'] == ticker]['DollarVolume'].values[0],
                    'Close': current['Close'],
                    'SMA_10': current['SMA_10']
                })
    except: continue

# 6. SELECT BEST 20 (DIVERSIFIED)
if candidates:
    # Separate lists
    bull_raw = [c for c in candidates if c['Signal'] == 'BULLISH']
    bear_raw = [c for c in candidates if c['Signal'] == 'BEARISH']
    
    # Apply Correlation Filter
    print("4/4: Removing Correlated Duplicates (Correlation > 0.85)...")
    final_bulls = filter_correlated_assets(bull_raw, data, max_items=10)
    final_bears = filter_correlated_assets(bear_raw, data, max_items=10)
    
    # Convert to DF for display
    bulls_df = pd.DataFrame(final_bulls)
    bears_df = pd.DataFrame(final_bears)
    
    # Create Lists
    bull_tickers = [x['Ticker'] for x in final_bulls]
    bear_tickers = [x['Ticker'] for x in final_bears]
    
    print("\n" + "="*80)
    print("TRADINGVIEW WATCHLIST STRINGS (DIVERSIFIED - NO DUPLICATES)")
    print("="*80)
    
    print(f"\n>>> BULLISH WATCHLIST -Credit Puts (Max 10 Unique):")
    print(", ".join(bull_tickers) if bull_tickers else "None found.")
    
    print(f"\n>>> BEARISH WATCHLIST -Credit Calls (Max 10 Unique):")
    print(", ".join(bear_tickers) if bear_tickers else "None found.")
    
    print("\n" + "="*80)
    print("DETAILED RESULTS")
    print("="*80)
    
    display_cols = ['Ticker', 'Signal', 'Close', 'SMA_10']
    
    if not bulls_df.empty:
        print(bulls_df[display_cols].to_string(index=False))
    if not bears_df.empty:
        print(bears_df[display_cols].to_string(index=False))

else:
    print("No trades found matching criteria today.")

# 7. PRINT RULES
print("\n" + "="*80)
print("⚠️ TRADE MANAGEMENT RULES")
print("="*80)
print("1. ENTRY:")
print("   - Expiration: ~9 Days to Expiration (DTE).")
print("   - Safety: No Earnings within 15 Days. Uncorrelated Assets.")
print("   - Strike: Sell strike with ~$0.20 - $0.25 credit.")
print("   - Spread Width: $1.00 (or $2.50/5.00 for larger stocks).")
print("\n2. STOP LOSS (THE EDGE):")
print("   - Set price alert at SOLD STRIKE.")
print("   - IF Stock Price CROSSES/CLOSES past Sold Strike -> CLOSE THE TRADE.")
print("="*80)